<hr style="border:1px solid #000000;">

# Verificação do Ambiente - Curso GEOTEC Aula 17

<hr style="border:1px solid #000000;">

Este notebook verifica se o ambiente Python, bibliotecas, banco de dados e tabelas estão configurados corretamente.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys

# Lista de bibliotecas necessárias
LIBRARIES = [
    'pandas',
    'geopandas',
    'shapely',
    'fiona',
    'pyproj',
    'sqlalchemy',
    'psycopg2',
    'geoalchemy2',
    'matplotlib',
    'folium',
    'cartopy',
    'jupyter',
    'numpy'
]

def check_python():
    print(f"Python: {sys.version}")
    if sys.version_info < (3, 8):
        print("⚠️ Python 3.8 ou superior é recomendado")
    else:
        print("Python OK")

def check_libraries():
    print("\nBibliotecas Python:")
    for lib in LIBRARIES:
        try:
            __import__(lib)
            print(f"✓ {lib}")
        except ImportError:
            print(f"X {lib} - NÃO INSTALADA")
            print(f"   Instale com: pip install {lib}")

def check_postgresql():
    print("\nPostgreSQL:")
    try:
        import psycopg2
        conn = psycopg2.connect(
            host='localhost',
            port=5432,
            database='postgres',
            user='postgres',
            password='postgres'
        )
        conn.close()
        print("✓ PostgreSQL rodando localmente (porta 5432)")
    except Exception:
        print("X Não foi possível conectar ao PostgreSQL local")
        print("X Verifique se o PostgreSQL está instalado e rodando")

def check_postgis():
    print("\nPostGIS:")
    try:
        import psycopg2
        conn = psycopg2.connect(
            host='localhost',
            port=5432,
            database='geotec',
            user='postgres',
            password='postgres'
        )
        with conn.cursor() as cur:
            cur.execute("SELECT PostGIS_version();")
            version = cur.fetchone()[0]
            print(f"✓ PostGIS OK: {version}")
        conn.close()
    except Exception:
        print("X PostGIS não detectado")

print("=" * 60)
print("Verificação do Ambiente - Curso GEOTEC")
print("=" * 60)

check_python()
check_libraries()
check_postgresql()
check_postgis()

Verificação do Ambiente - Curso GEOTEC
Python: 3.13.12 | packaged by conda-forge | (main, Feb  5 2026, 05:41:12) [MSC v.1944 64 bit (AMD64)]
Python OK

Bibliotecas Python:
✓ pandas
✓ geopandas
✓ shapely
✓ fiona
✓ pyproj
✓ sqlalchemy
✓ psycopg2
✓ geoalchemy2
✓ matplotlib
✓ folium
✓ cartopy
✓ jupyter
✓ numpy

PostgreSQL:
✓ PostgreSQL rodando localmente (porta 5432)

PostGIS:
✓ PostGIS OK: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


<hr style="border:1px solid #000000;">

# Insert Projeção Albers SRID 97823

<hr style="border:1px solid #000000;">

In [3]:
import psycopg2

def insert_projection():
    print("\nInserindo projeção personalizada (SRID 97823)...")

    sql = """
    BEGIN;

    INSERT INTO public.spatial_ref_sys (
      srid,
      auth_name,
      auth_srid,
      srtext,
      proj4text
    )
    VALUES (
      97823,
      'IBGE',
      97823,
      'PROJCS["Conica_Equivalente_de_Albers_Brasil",
        GEOGCS["GCS_SIRGAS2000",
          DATUM["D_SIRGAS2000",
            SPHEROID["Geodetic_Reference_System_of_1980",6378137,298.2572221009113]],
          PRIMEM["Greenwich",0],
          UNIT["Degree",0.017453292519943295]],
        PROJECTION["Albers"],
        PARAMETER["standard_parallel_1",-2],
        PARAMETER["standard_parallel_2",-22],
        PARAMETER["latitude_of_origin",-12],
        PARAMETER["central_meridian",-54],
        PARAMETER["false_easting",5000000],
        PARAMETER["false_northing",10000000],
        UNIT["Meter",1]]',
      '+proj=aea +lat_0=-12 +lon_0=-54 +lat_1=-2 +lat_2=-22 +x_0=5000000 +y_0=10000000 +ellps=GRS80 +units=m +no_defs'
    )
    ON CONFLICT (srid) DO NOTHING;

    COMMIT;
    """

    try:
        conn = psycopg2.connect(
            host='localhost',
            port=5432,
            database='geotec',
            user='postgres',
            password='postgres'
        )
        cur = conn.cursor()
        cur.execute(sql)
        conn.commit()

        print("✓ Projeção inserida com sucesso (ou já existia)")

        cur.close()
        conn.close()

    except Exception as e:
        print("X Erro ao inserir projeção:")
        print(e)

insert_projection()


Inserindo projeção personalizada (SRID 97823)...
✓ Projeção inserida com sucesso (ou já existia)


<hr style="border:1px solid #000000;">

# Importação de Dados

<hr style="border:1px solid #000000;">

In [2]:
# -*- coding: utf-8 -*-
# =============================================================================
# IMPORTACAO DE DADOS GPKG/SHP/CSV -> PostgreSQL  |  PROJETO GEOTEC
# =============================================================================
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============================================================================
# CONFIGURACOES - ALTERE CONFORME NECESSARIO
# =============================================================================
PASTA_DADOS = Path(r"C:\Temp")

# =============================================================================
# ARQUIVOS (Variáveis sem extensão para permitir GPKG ou SHP dinamicamente)
# =============================================================================
APPS            = "es_pi_apps_sicar"
CONSOLIDADA     = "es_pi_area_consolidada_sicar"
IMOVEL          = "es_pi_area_imovel_sicar"
POUSIO          = "es_pi_area_pousio_sicar"
ASV             = "es_pi_asv_sinaflor"
HIDRO           = "es_pi_hidrografia_sicar"
MUNI            = "es_pi_municipios_ibge_2025"
RESERVA         = "es_pi_reserva_legal_sicar"
RESID_CAAT      = "es_pi_residual_biome_caatinga"
RESID_CERR      = "es_pi_residual_biome_cerrado"
SERVIDAO        = "es_pi_servidao_administrativa_sicar"
USO_REST        = "es_pi_uso_restrito_sicar"
VEG_NAT         = "es_pi_vegetacao_nativa_sicar"
DEF_CAAT        = "es_pi_yearly_deforestation_biome_caatinga"
DEF_CERR        = "es_pi_yearly_deforestation_biome_cerrado"
AMAZONIA        = "pa_br_amazonia_legal"
MODULOS         = "pa_br_modulosfiscais_incra.csv"  # Mantido com extensão por ser CSV fixo

# =============================================================================
# CONTROLE DE IMPORTACAO
# =============================================================================
RECRIAR_TABELAS = False  # Se False, pula as que já existem no banco

# =============================================================================
# CREDENCIAIS DO BANCO LOCAL (geotec)
# =============================================================================
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_USER     = "postgres"
DB_PASSWORD = "postgres"
DB_NAME     = "geotec"

# =============================================================================
# MAPEAMENTO: arquivo -> (schema_generico, tabela)
# =============================================================================
IMPORTACOES = [
    # SICAR / CAR
    (APPS,            "car",   "es_pi_apps_sicar"),
    (CONSOLIDADA,     "car",   "es_pi_area_consolidada_sicar"),
    (IMOVEL,          "car",   "es_pi_area_imovel_sicar"),
    (POUSIO,          "car",   "es_pi_area_pousio_sicar"),
    (HIDRO,           "car",   "es_pi_hidrografia_sicar"),
    (RESERVA,         "car",   "es_pi_reserva_legal_sicar"),
    (SERVIDAO,        "car",   "es_pi_servidao_administrativa_sicar"),
    (USO_REST,        "car",   "es_pi_uso_restrito_sicar"),
    (VEG_NAT,         "car",   "es_pi_vegetacao_nativa_sicar"),

    # INPE (Antigo MMA)
    (DEF_CAAT,        "inpe",  "es_pi_yearly_deforestation_biome_caatinga"),
    (DEF_CERR,        "inpe",  "es_pi_yearly_deforestation_biome_cerrado"),
    (RESID_CAAT,      "inpe",  "es_pi_residual_biome_caatinga"),
    (RESID_CERR,      "inpe",  "es_pi_residual_biome_cerrado"),

    # IBGE
    (AMAZONIA,        "ibge",  "pa_br_amazonia_legal"),
    (MUNI,            "ibge",  "es_pi_municipios_ibge_2025"),

    # INCRA
    (MODULOS,         "incra", "pa_br_modulosfiscais_incra"),

    # IBAMA (Antigo Sinaflor)
    (ASV,             "ibama", "es_pi_asv_sinaflor"),
]

# =============================================================================
# FUNCOES
# =============================================================================
def conectar_banco():
    return create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

def criar_schemas(engine):
    schemas = list(set([imp[1] for imp in IMPORTACOES]))
    with engine.connect() as conn:
        for s in schemas:
            conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {s}"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print(f"[OK] Schemas prontos: {', '.join(schemas)}")

def tabela_existe(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f'SELECT 1 FROM "{schema}"."{tabela}" LIMIT 1'))
            return True
    except Exception:
        return False

def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            r = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{tabela}"'))
            return r.fetchone()[0]
    except Exception:
        return 0

def importar_arquivo(engine, caminho, schema, tabela):
    print(f"  Lendo: {caminho.name}")
    extensao = caminho.suffix.lower()
    
    # Lógica para arquivos CSV
    if extensao == '.csv':
        df = pd.read_csv(caminho)
        print(f"  Registros (CSV): {len(df)}")
        df.to_sql(name=tabela, con=engine, schema=schema, if_exists="replace", index=False)
    
    # Lógica para arquivos Espaciais (GPKG ou SHP)
    else:
        gdf = gpd.read_file(str(caminho), engine='pyogrio')
        print(f"  Registros ({extensao.upper()[1:]}): {len(gdf)}")

        if not gdf.empty and gdf.geometry is not None:
            # Renomeia a coluna de geometria para 'geom' antes de subir
            gdf = gdf.rename_geometry('geom')
            
            if gdf.crs is None:
                gdf = gdf.set_crs(epsg=4674)
            else:
                gdf = gdf.to_crs(epsg=4674)
            
            gdf.to_postgis(
                name=tabela, con=engine, schema=schema,
                if_exists="replace", index=False
            )
        else:
            df = pd.DataFrame(gdf).drop(columns=['geometry', 'geom'], errors='ignore')
            df.to_sql(name=tabela, con=engine, schema=schema, if_exists="replace", index=False)

    print(f"  OK -> {schema}.{tabela}")

# =============================================================================
# MAIN
# =============================================================================
def main():
    print("=" * 65)
    print("IMPORTACAO DE DADOS GEOPROCESSAMENTO - PROJETO GEOTEC")
    print("=" * 65)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    criar_schemas(engine)

    for i, (nome_base, schema, tabela) in enumerate(IMPORTACOES, 1):
        print(f"\n[{i}/{len(IMPORTACOES)}] {schema}.{tabela}")
        
        # Identifica o caminho correto baseado na existência do arquivo
        if nome_base.endswith('.csv'):
            caminho = PASTA_DADOS / nome_base
        else:
            caminho_gpkg = PASTA_DADOS / f"{nome_base}.gpkg"
            caminho_shp = PASTA_DADOS / f"{nome_base}.shp"
            
            if caminho_gpkg.exists():
                caminho = caminho_gpkg
            elif caminho_shp.exists():
                caminho = caminho_shp
            else:
                print(f"  [ERRO] Arquivo (.gpkg ou .shp) nao encontrado para: {nome_base}")
                continue

        if tabela_existe(engine, schema, tabela) and not RECRIAR_TABELAS:
            n = contar_registros(engine, schema, tabela)
            print(f"  [EXISTE] {schema}.{tabela} ({n} registros) - mantido")
            continue

        importar_arquivo(engine, caminho, schema, tabela)

    print("\n" + "=" * 65)
    print("RESUMO FINAL")
    print("=" * 65)
    for _, schema, tabela in IMPORTACOES:
        n = contar_registros(engine, schema, tabela)
        print(f"  {schema}.{tabela}: {n} registros")

    engine.dispose()
    print("\n[OK] Importacao concluida!")


if __name__ == "__main__":
    main()

IMPORTACAO DE DADOS GEOPROCESSAMENTO - PROJETO GEOTEC
[OK] Schemas prontos: inpe, car, ibge, incra, ibama

[1/17] car.es_pi_apps_sicar
  [EXISTE] car.es_pi_apps_sicar (245833 registros) - mantido

[2/17] car.es_pi_area_consolidada_sicar
  [EXISTE] car.es_pi_area_consolidada_sicar (227759 registros) - mantido

[3/17] car.es_pi_area_imovel_sicar
  [EXISTE] car.es_pi_area_imovel_sicar (332628 registros) - mantido

[4/17] car.es_pi_area_pousio_sicar
  [EXISTE] car.es_pi_area_pousio_sicar (26630 registros) - mantido

[5/17] car.es_pi_hidrografia_sicar
  [EXISTE] car.es_pi_hidrografia_sicar (44912 registros) - mantido

[6/17] car.es_pi_reserva_legal_sicar
  [EXISTE] car.es_pi_reserva_legal_sicar (284301 registros) - mantido

[7/17] car.es_pi_servidao_administrativa_sicar
  [EXISTE] car.es_pi_servidao_administrativa_sicar (95211 registros) - mantido

[8/17] car.es_pi_uso_restrito_sicar
  [EXISTE] car.es_pi_uso_restrito_sicar (3719 registros) - mantido

[9/17] car.es_pi_vegetacao_nativa_sicar
